# 05 — Model Verification & Export Tests

**Goal:** Load the exported artifacts (`encoder.pkl`, `xgb_delay_model.pkl`, `xgb_budget_model.pkl`) and perform a dummy prediction to verify they are ready for the FastAPI backend.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings

warnings.filterwarnings('ignore')

## 1. Load Artifacts

In [ ]:
print("Loading artifacts from ../model_artifacts/...")

encoder = joblib.load('../model_artifacts/encoder.pkl')
config = joblib.load('../model_artifacts/feature_config.pkl')
delay_model = joblib.load('../model_artifacts/xgb_delay_model.pkl')
budget_model = joblib.load('../model_artifacts/xgb_budget_model.pkl')

print("\u2705 All artifacts loaded successfully.")

## 2. Simulate API Request (Dummy Data)

In [ ]:
# This mimics the payload the FastAPI backend will receive
dummy_input = {
    'Region': ['Cordillera Administrative Region'],
    'Province': ['Apayao'],
    'Municipality': ['Calanasan'],
    'TypeOfWork': ['Construction of Flood Mitigation Structure'],
    'Contractor': ['ASC CONSTRUCTION & CONCRETE PRODUCTS'],
    'MainIsland': ['Luzon'],
    'ApprovedBudgetForContract': [96500000.0],
    'ContractorCount': [1],
    'FundingYear': [2024]
}

df_dummy = pd.DataFrame(dummy_input)
print("Raw Input:")
display(df_dummy)

## 3. Apply Encoding

In [ ]:
# Extract categories to encode
cat_cols = config['cat_cols']
encoded_cols = config['encoded_cols']

# Ensure the dummy dataframe has all categorical columns (even if some aren't used in a specific model)
for col in cat_cols:
    if col not in df_dummy.columns:
        df_dummy[col] = 'Unknown'

df_dummy[encoded_cols] = encoder.transform(df_dummy[cat_cols])

print("Encoded Input:")
display(df_dummy[encoded_cols])

## 4. Run Predictions

In [ ]:
# --- Delay Prediction ---
X_delay = df_dummy[config['delay_features']]
delay_pred = delay_model.predict(X_delay)[0]
delay_prob = delay_model.predict_proba(X_delay)[0][1]

print("\n=== Delay Prediction ===")
print(f"Is Delayed?: {'Yes' if delay_pred == 1 else 'No'}")
print(f"Probability of Delay: {delay_prob:.2%}")

# --- Budget Prediction ---
X_budget = df_dummy[config['budget_features']]
budget_pred = budget_model.predict(X_budget)[0]

print("\n=== Budget Prediction ===")
print(f"Estimated Contract Cost: PHP {budget_pred:,.2f}")
print(f"Variance from Approved Budget: PHP {budget_pred - dummy_input['ApprovedBudgetForContract'][0]:,.2f}")

## 5. Conclusion
Models are working as expected and are ready to be integrated into the FastAPI backend (`backend/app/routes/predict.py`).